In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 岭回归建模
print("=" * 80)
print("房屋价格预测模型 - 岭回归 (Ridge Regression)")
print("=" * 80)

# 读取数据
df = pd.read_csv('house_price_final_train_dataset3.csv')
print(f"数据形状: {df.shape}")

# 检查目标变量
if 'lnPrice' not in df.columns and 'Price' in df.columns:
    df['lnPrice'] = np.log(df['Price'])
    print("已创建lnPrice变量")

# 特征工程 - 创建衍生变量
print("\n执行特征工程...")

# 1. 创建幂次项
continuous_vars_for_power = [
    'area', 'building_age', 'greening_rate', 'plot_ratio', 
    'property_fee_avg', 'room_count', 'total_floor', 'gas_fee_avg'
]

for var in continuous_vars_for_power:
    if var in df.columns:
        square_term = f'{var}^2'
        df[square_term] = df[var] ** 2
        print(f"已创建平方项: {square_term}")

# 2. 创建城市哑变量
if '城市' in df.columns:
    for city_num in range(12):
        col_name = f'city_{city_num}'
        df[col_name] = (df['城市'] == city_num).astype(int)
    print("已创建城市哑变量")

# 3. 创建城市距离变量
if '城市' in df.columns and '距市中心距离_km' in df.columns:
    for city_num in range(12):
        col_name = f'city_dist_{city_num}'
        df[col_name] = 0
        df.loc[df['城市'] == city_num, col_name] = df.loc[df['城市'] == city_num, '距市中心距离_km']
    print("已创建城市距离变量")

# 定义基准模型变量
base_vars = [
    # 装修类型
    'decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他',
    # 楼层位置
    'high_dummy', 'middle_dummy', 'low_dummy', 'basement_dummy', 'top_dummy', 'bottom_dummy',
    # 房屋基本信息
    'total_floor', 'area', 'room_count', 'hall_count', 
    # 朝向
    'south_dummy', 'north_south_dummy',
    # 交易时间
    'trans_2024', 'trans_2025',
    # 梯户信息
    '梯数', '户数', 'elevator_yes',
    # 交易类型
    'transaction_commercial', 'transaction_non_commercial',
    # 用途类型
    'usage_commercial_office', 'usage_commercial_residential', 'usage_high_end_residential',
    'usage_ordinary_residential', 'usage_other_special',
    # 房龄
    'house_age_over_2_years', 'house_age_over_5_years', 'house_age_under_2_years',
    # 交通和位置
    'subway', '距市中心距离_km',
    # 建筑信息
    'building_age', 'household_total', 'building_total',
    'greening_rate', 'plot_ratio',
    # 结构材料
    'structure_material_brick_concrete', 'structure_material_brick_wood', 'structure_material_frame',
    'structure_material_mixed', 'structure_material_steel', 'structure_material_steel_concrete',
    'structure_material_unknown',
    # 费用信息
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg',
    # 水电暖类型
    'water_civil', 'water_commercial',
    'heating_central', 'heating_self',
    'electricity_civil', 'electricity_commercial',
    # 停车位
    'parking_spots',
    # 周边设施
    'surrounding_hospital', 'surrounding_university', 'surrounding_school', 'surrounding_supermarket',
    'property_phone_yes',
    # 城市信息
    '城市',  # 原始城市编码
    # 城市哑变量
    'city_0', 'city_1', 'city_2', 'city_3', 'city_4', 'city_5',
    'city_6', 'city_7', 'city_8', 'city_9', 'city_10', 'city_11',
    # 城市距离变量
    'city_dist_0', 'city_dist_1', 'city_dist_2', 'city_dist_3', 'city_dist_4', 'city_dist_5',
    'city_dist_6', 'city_dist_7', 'city_dist_8', 'city_dist_9', 'city_dist_10', 'city_dist_11',
    # 面积平方项
    'area^2'
]

# 添加城市哑变量
city_dummies = [f'city_{i}' for i in range(12)]
base_vars.extend(city_dummies)

# 添加幂次项
for var in continuous_vars_for_power:
    square_term = f'{var}^2'
    if square_term in df.columns and square_term not in base_vars:
        base_vars.append(square_term)

# 检查变量存在性
existing_base_vars = [col for col in base_vars if col in df.columns]
print(f"使用的变量数量: {len(existing_base_vars)}")

# 准备数据
X = df[existing_base_vars]
y = df['lnPrice']

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"训练集大小: {X_train.shape}, 测试集大小: {X_test.shape}")

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用交叉验证选择最佳alpha值
print("\n使用交叉验证选择最佳alpha值...")
alphas = np.logspace(-2, 4, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)
print(f"最佳alpha值: {ridge_cv.alpha_:.6f}")

# 训练最终岭回归模型
ridge_model = Ridge(alpha=ridge_cv.alpha_, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

# 预测函数
def predict_and_evaluate(model, X, y, scaler=None, is_scaled=False):
    if not is_scaled and scaler is not None:
        X = scaler.transform(X)
    
    y_pred_ln = model.predict(X)
    y_pred = np.exp(y_pred_ln)
    y_true = np.exp(y)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return mae, rmse, r2

# 样本内性能
train_mae, train_rmse, train_r2 = predict_and_evaluate(
    ridge_model, X_train_scaled, y_train, is_scaled=True
)

# 样本外性能
test_mae, test_rmse, test_r2 = predict_and_evaluate(
    ridge_model, X_test, y_test, scaler=scaler
)

# 6折交叉验证
print("\n进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae, cv_scores_rmse, cv_scores_r2 = [], [], []

for train_idx, val_idx in kf.split(X):
    X_cv_train, X_cv_val = X.iloc[train_idx], X.iloc[val_idx]
    y_cv_train, y_cv_val = y.iloc[train_idx], y.iloc[val_idx]
    
    scaler_cv = StandardScaler()
    X_cv_train_scaled = scaler_cv.fit_transform(X_cv_train)
    X_cv_val_scaled = scaler_cv.transform(X_cv_val)
    
    cv_model = Ridge(alpha=ridge_cv.alpha_, random_state=42)
    cv_model.fit(X_cv_train_scaled, y_cv_train)
    
    cv_mae, cv_rmse, cv_r2 = predict_and_evaluate(
        cv_model, X_cv_val_scaled, y_cv_val, is_scaled=True
    )
    
    cv_scores_mae.append(cv_mae)
    cv_scores_rmse.append(cv_rmse)
    cv_scores_r2.append(cv_r2)

cv_mae_mean = np.mean(cv_scores_mae)
cv_rmse_mean = np.mean(cv_scores_rmse)
cv_r2_mean = np.mean(cv_scores_r2)

# 输出结果
print("\n" + "=" * 80)
print("岭回归模型性能指标")
print("=" * 80)

print(f"\n性能指标表格:")
print("-" * 65)
print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'Cross-validation':<18}")
print("-" * 65)
print(f"{'R²':<15} {train_r2:.4f}{'':<8} {test_r2:.4f}{'':<10} {cv_r2_mean:.4f}")
print(f"{'MAE':<15} {train_mae:.2f}{'':<8} {test_mae:.2f}{'':<10} {cv_mae_mean:.2f}")
print(f"{'RMSE':<15} {train_rmse:.2f}{'':<8} {test_rmse:.2f}{'':<10} {cv_rmse_mean:.2f}")
print("-" * 65)

print(f"\n详细统计信息:")
print(f"样本数量 (训练集): {len(X_train)}")
print(f"样本数量 (测试集): {len(X_test)}")
print(f"变量数量: {X_train.shape[1]}")
print(f"最佳alpha值: {ridge_cv.alpha_:.6f}")

# 保存模型结果
model_results = {
    'Ridge': {
        'model': ridge_model,
        'scaler': scaler,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2': cv_r2_mean,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'cv_mae': cv_mae_mean,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse_mean,
        'alpha': ridge_cv.alpha_,
        'feature_names': existing_base_vars
    }
}

print("\n" + "=" * 80)
print("岭回归模型完成")
print("=" * 80)

# 简化的测试集预测函数
def predict_test_set_ridge_simple(model_results, test_data_path='house_price_final_test_dataset.csv'):
    print("\n" + "=" * 80)
    print("测试集预测 - 岭回归模型")
    print("=" * 80)
    
    # 读取测试集数据
    test_df = pd.read_csv(test_data_path)
    print(f"原始测试集形状: {test_df.shape}")
    
    # 获取模型和标准化器
    ridge_model = model_results['Ridge']['model']
    scaler = model_results['Ridge']['scaler']
    feature_names = model_results['Ridge']['feature_names']
    
    # 特征工程
    print("\n执行特征工程...")
    
    # 创建幂次项
    for var in continuous_vars_for_power:
        if var in test_df.columns:
            square_term = f'{var}^2'
            test_df[square_term] = test_df[var] ** 2
    
    # 创建城市哑变量
    if '城市' in test_df.columns:
        for city_num in range(12):
            col_name = f'city_{city_num}'
            test_df[col_name] = (test_df['城市'] == city_num).astype(int)
    
    # 创建城市距离变量
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        for city_num in range(12):
            col_name = f'city_dist_{city_num}'
            test_df[col_name] = 0
            test_df.loc[test_df['城市'] == city_num, col_name] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
    
    # 处理缺失值 - 使用训练集统计量
    print("处理缺失值...")
    for col in feature_names:
        if col in test_df.columns and test_df[col].isnull().sum() > 0:
            if col in df.columns:
                if df[col].dtype in ['float64', 'int64']:
                    # 使用训练集中位数填充
                    median_val = df[col].median()
                    test_df[col] = test_df[col].fillna(median_val)
                else:
                    # 使用训练集众数填充
                    mode_val = df[col].mode()[0] if not df[col].mode().empty else 0
                    test_df[col] = test_df[col].fillna(mode_val)
    
    # 确保测试集包含所有训练模型的特征
    missing_features = set(feature_names) - set(test_df.columns)
    if missing_features:
        for feature in missing_features:
            test_df[feature] = 0
    
    # 选择特征并标准化
    X_test_final = test_df[feature_names]
    X_test_scaled = scaler.transform(X_test_final)
    
    # 进行预测
    print("\n使用岭回归模型进行预测...")
    y_test_pred_ln = ridge_model.predict(X_test_scaled)
    y_test_pred = np.exp(y_test_pred_ln)
    
    # 创建结果DataFrame
    if 'ID' in test_df.columns:
        result_df = pd.DataFrame({
            'ID': test_df['ID'],
            'Price': y_test_pred
        })
    else:
        result_df = pd.DataFrame({
            'ID': test_df.index,
            'Price': y_test_pred
        })
    
    # 保存预测结果
    output_test_path = 'house_price_test_predictions_ridge.csv'
    result_df.to_csv(output_test_path, index=False, float_format='%.2f')
    print(f"\n预测结果已保存到: {output_test_path}")
    print(f"结果文件包含 {len(result_df)} 条预测记录")
    
    # 显示结果统计
    print("\n预测结果统计信息:")
    print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
    print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
    print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")
    
    return result_df

# 运行测试集预测
predict_test_set_ridge_simple(model_results)

房屋价格预测模型 - 岭回归 (Ridge Regression)
数据形状: (103871, 90)

执行特征工程...
已创建平方项: area^2
已创建平方项: building_age^2
已创建平方项: greening_rate^2
已创建平方项: plot_ratio^2
已创建平方项: property_fee_avg^2
已创建平方项: room_count^2
已创建平方项: total_floor^2
已创建平方项: gas_fee_avg^2
已创建城市哑变量
已创建城市距离变量
使用的变量数量: 105
训练集大小: (83096, 105), 测试集大小: (20775, 105)

使用交叉验证选择最佳alpha值...
最佳alpha值: 1.206793

进行6折交叉验证...

岭回归模型性能指标

性能指标表格:
-----------------------------------------------------------------
Metrics         In sample    Out of sample  Cross-validation  
-----------------------------------------------------------------
R²              0.6939         0.6762           0.6800
MAE             655540.84         665417.84           658229.56
RMSE            1398683.33         1452154.19           1424385.24
-----------------------------------------------------------------

详细统计信息:
样本数量 (训练集): 83096
样本数量 (测试集): 20775
变量数量: 105
最佳alpha值: 1.206793

岭回归模型完成

测试集预测 - 岭回归模型
原始测试集形状: (34017, 64)

执行特征工程...
处理缺失值...

使用岭回归模型进行预测...

预测结果已保存到: ho

,ID,Price
0,1000000,1.325506e+07
1,1000001,4.470737e+06
2,1000002,7.388507e+06
3,1000003,2.842291e+06
4,1000004,6.015457e+06
...,...,...
34012,1034012,1.239847e+06
34013,1034013,4.144958e+05
34014,1034014,7.323281e+05
34015,1034015,7.341197e+05
